In [1]:
!pip install google-play-scraper

In [3]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time

# All Indian startup app IDs
apps = {
    'Zomato': 'com.application.zomato',
    'Swiggy': 'in.swiggy.android',
    'Blinkit': 'com.grofers.customerapp',
    'CRED': 'com.dreamplug.androidapp',
    'Zepto': 'com.zeptoconsumerapp',
    'Meesho': 'com.meesho.supply',
    'PhonePe': 'com.phonepe.app',
    'Paytm': 'net.one97.paytm',
    'Ola': 'com.olacabs.customer',
    'Uber': 'com.ubercab'
}

all_reviews = []

for brand, app_id in apps.items():
    try:
        print(f"Scraping {brand}...")
        result, _ = reviews(
            app_id,
            lang='en',
            country='in',
            sort=Sort.NEWEST,
            count=50000
        )
        df = pd.DataFrame(result)
        df['brand'] = brand
        all_reviews.append(df)
        print(f" {brand}: {len(df)} reviews collected")
        time.sleep(2)  # Wait 2 seconds between each brand
    except Exception as e:
        print(f"{brand} failed: {e}")

# Combine all brands
final_df = pd.concat(all_reviews, ignore_index=True)
print(f"\n Total reviews collected: {len(final_df)}")
final_df.to_csv("raw_reviews.csv", index=False)
print("Saved to raw_reviews.csv!")

Scraping Zomato...
 Zomato: 50000 reviews collected
Scraping Swiggy...
 Swiggy: 50000 reviews collected
Scraping Blinkit...
 Blinkit: 50000 reviews collected
Scraping CRED...
 CRED: 50000 reviews collected
Scraping Zepto...
 Zepto: 50000 reviews collected
Scraping Meesho...
 Meesho: 50000 reviews collected
Scraping PhonePe...
 PhonePe: 50000 reviews collected
Scraping Paytm...
 Paytm: 50000 reviews collected
Scraping Ola...
 Ola: 50000 reviews collected
Scraping Uber...
 Uber: 50000 reviews collected

 Total reviews collected: 500000
Saved to raw_reviews.csv!


In [10]:
df = pd.read_csv("raw_reviews.csv")
print("Total rows:", len(df))
print("\nColumns:", df.columns.tolist())
print("\nReviews per brand:")
print(df['brand'].value_counts())
print("\nSample data:")
print(df.head(3))

Total rows: 500000

Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'brand']

Reviews per brand:
brand
Zomato     50000
Swiggy     50000
Blinkit    50000
CRED       50000
Zepto      50000
Meesho     50000
PhonePe    50000
Paytm      50000
Ola        50000
Uber       50000
Name: count, dtype: int64

Sample data:
                               reviewId       userName  \
0  30d0934d-d3ff-494e-8151-0fd85ec9cdf6  A Google user   
1  047ee158-034a-4438-92a6-65e53934c32e  A Google user   
2  31480dca-40c8-4ceb-befe-9ce81155bf27  A Google user   

                                           userImage  \
0  https://play-lh.googleusercontent.com/EGemoI2N...   
1  https://play-lh.googleusercontent.com/EGemoI2N...   
2  https://play-lh.googleusercontent.com/EGemoI2N...   

                                             content  score  thumbsUpCount  \
0                                         

In [7]:
# Keep only useful columns
df_clean = df[['reviewId', 'userName', 'content', 'score', 
               'thumbsUpCount', 'at', 'brand', 'appVersion']]

# Rename columns
df_clean.columns = ['review_id', 'user_name', 'review_text', 
                    'rating', 'thumbs_up', 'date', 'brand', 'app_version']

# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['review_id'])

print("After cleaning:", len(df_clean))
print(df_clean.head(3))

# Save cleaned version
df_clean.to_csv("cleaned_reviews.csv", index=False)
print("Saved to cleaned_reviews.csv!")

After cleaning: 500000
                              review_id      user_name  \
0  30d0934d-d3ff-494e-8151-0fd85ec9cdf6  A Google user   
1  047ee158-034a-4438-92a6-65e53934c32e  A Google user   
2  31480dca-40c8-4ceb-befe-9ce81155bf27  A Google user   

                                         review_text  rating  thumbs_up  \
0                                             good 😊       2          0   
1                                               good       5          0   
2  totally waste my first order come 1:30 cold no...       1          0   

                  date   brand app_version  
0  2026-06-18 00:11:36  Zomato      19.6.8  
1  2026-06-18 00:10:51  Zomato      19.6.8  
2  2026-06-18 00:09:56  Zomato      19.7.2  
Saved to cleaned_reviews.csv!


In [11]:
df = pd.read_csv("cleaned_reviews.csv")

# Create sentiment from rating
def get_sentiment(rating):
    if rating >= 4:
        return 'Positive'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Negative'

df['sentiment'] = df['rating'].apply(get_sentiment)

# Check distribution
print("Sentiment Distribution:")
print(df['sentiment'].value_counts())
print("\nSentiment per brand:")
print(df.groupby(['brand', 'sentiment']).size().unstack(fill_value=0))

# Save
df.to_csv("cleaned_reviews.csv", index=False)
print("\n Saved with sentiment labels!")

Sentiment Distribution:
sentiment
Positive    362074
Negative    122104
Neutral      15822
Name: count, dtype: int64

Sentiment per brand:
sentiment  Negative  Neutral  Positive
brand                                 
Blinkit        9283     2253     38464
CRED           9972     1193     38835
Meesho         8678      865     40457
Ola           24420     1553     24027
Paytm          7546     1419     41035
PhonePe        4525     2032     43443
Swiggy        17419     1980     30601
Uber          12135     1344     36521
Zepto         19672     1066     29262
Zomato         8454     2117     39429

 Saved with sentiment labels!


In [12]:
import pandas as pd
import re

df = pd.read_csv("cleaned_reviews.csv")

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)        # remove links
    text = re.sub(r'[^a-zA-Z\s]', '', text)           # remove emojis, numbers, symbols
    text = re.sub(r'\s+', ' ', text).strip()          # remove extra spaces
    return text

df['clean_text'] = df['review_text'].apply(clean_text)

# Remove empty reviews
df = df[df['clean_text'].str.len() > 3]

print("Total after text cleaning:", len(df))
print("\nSample cleaned text:")
print(df[['review_text', 'clean_text']].head(5))

df.to_csv("cleaned_reviews.csv", index=False)
print("\n Saved!")

Total after text cleaning: 475220

Sample cleaned text:
                                         review_text  \
0                                             good 😊   
1                                               good   
2  totally waste my first order come 1:30 cold no...   
3                                        excellent 👍   
5                                          right now   

                                          clean_text  
0                                               good  
1                                               good  
2  totally waste my first order come cold no cust...  
3                                          excellent  
5                                          right now  

 Saved!
